# 00. Navigate the package

Start here. This notebook is the entry point to the example suite:

1. **Install** the project (one of three paths below).
2. **Sanity-check** the install (Java, bridge, CLI on PATH).
3. **Tour the CLI**: the six subcommands and how to discover their flags.
4. **Tour the repo**: where data, agents, experiments, examples live.

Once this notebook is green end-to-end, every other notebook in `examples/`
assumes the environment is set up and dives straight into its task.

Run from the repo root (`jupyter lab` in the project directory).

## 1. Install

Three setup paths shipped under [`setup/`](../setup/), pick one:

| Path | When | Command |
|---|---|---|
| [`setup/local.sh`](../setup/local.sh) | Dev on your laptop (macOS/Linux) | `bash setup/local.sh` |
| [`setup/cluster.sh`](../setup/cluster.sh) | HPC (CECI: Lyra, Manneback, ...) | `module load Python/3.10 Java/17.0.6 [CUDA/12.1.1]` then `bash setup/cluster.sh` |
| [`setup/docker.sh`](../setup/docker.sh) | Docker (CPU or GPU image) | `bash setup/docker.sh [cpu\|gpu\|both]` |

All three share the same helpers in [`setup/_common.sh`](../setup/_common.sh): rebuild the JNI bridge, install Python deps with the `[dev,tournament]` extras, fetch the RAISocketAI competition wheel (hash-verified). The cluster script additionally bootstraps a Python 3.6 venv for the UTS_Imass bot's BL_JPS pathfinding.

**Activate after setup**:

```bash
conda activate microrts_agent          # local.sh path
source cluster_venv/bin/activate       # cluster.sh path
# (Docker path: no activation, just docker run)
```

## 2. Sanity check

Verify the install is complete before moving on. Subsequent notebooks (01-06) do **not** repeat these checks; they assume this notebook is green.

In [ ]:
import importlib.util
import shutil

from microrts_agent.paths import PROJECT_ROOT

assert importlib.util.find_spec("microrts_agent"), (
    "microrts_agent not importable. Run bash setup/local.sh"
)
assert shutil.which("java"), "java not on PATH (need JDK 17+). On HPC: module load Java/17.0.6"
assert shutil.which("microrts-agent"), (
    "microrts-agent console script not on PATH. Activate the venv."
)

bridge_jar = PROJECT_ROOT / "microrts_agent" / "microrts" / "lib" / "bridge.jar"
assert bridge_jar.exists(), (
    f"bridge.jar missing at {bridge_jar}. Run bash microrts_agent/microrts/build_bridge.sh"
)

print("Sanity check OK")
print("Repo root :", PROJECT_ROOT)
print("Java      :", shutil.which("java"))
print("CLI       :", shutil.which("microrts-agent"))
print("Bridge    :", bridge_jar.relative_to(PROJECT_ROOT))

## 3. CLI tour

The package exposes **one** entry point: `microrts-agent <command>`. There are six commands. Every command supports `--help` for its full flag list.

In [ ]:
import subprocess

print(subprocess.run(["microrts-agent", "--help"], capture_output=True, text=True).stdout)

The output above lists every subcommand with its key flags and one usage example. Three of them have **nested sub-commands** (`tournament`, `bc`, `bench`, `analysis`); for those, `microrts-agent <command> --help` prints the sub-command tree. The two simple ones (`train`, `evaluate`) drop straight into their argparse output.

Each command is the subject of its own notebook in this folder:

- [`01_evaluate.ipynb`](01_evaluate.ipynb): `evaluate`
- [`02_train.ipynb`](02_train.ipynb): `train`
- [`03_tournament.ipynb`](03_tournament.ipynb): `tournament {run,parse,viz,analyze}`
- [`04_bc.ipynb`](04_bc.ipynb): `bc {generate,train}`
- [`05_bench.ipynb`](05_bench.ipynb): `bench {inference,head2head}`
- [`06_analysis.ipynb`](06_analysis.ipynb): `analysis {metrics,audit,params}`

## 4. Repository layout

Where to find things on disk:

In [ ]:
items = [
    (
        "microrts_agent/",
        "Importable Python package: envs, architectures, training loop, tournament, JNI bridge",
    ),
    ("data/agents/", "9 shipped trained agents (UECD family + GridNet + BC + BC-PPO)"),
    ("data/BC/", "BC teacher dataset (training/) + baseline numbers (baseline/)"),
    ("data/tournaments/", "Headline thesis tournaments (single_map + multi_map)"),
    ("data/ablation/", "Architecture (arch/) + feature (feat/) ablations"),
    ("data/recordings/", "36 mp4 clips of UECD-Best vs the field"),
    ("data/rush_collapse/", "Pre/post-collapse eval of UECD-SingleMap-Rushed"),
    ("data/generalization_probes/", "UECD-Best on unseen maps"),
    ("experiments/", "SLURM job scripts (single-map, multi-map, BC, eval, tournament, ablation)"),
    ("setup/", "local.sh + cluster.sh + docker.sh + shared _common.sh"),
    ("tests/", "115 smoke tests run by CI (~90s)"),
    ("dissertation/", "LaTeX thesis + figure generators (figs/figs-python/)"),
    ("cog-2026-paper/", "CoG 2026 short-paper submission"),
    ("outputs/", "Runtime outputs (gitignored): training runs, BC data, tournament results"),
]
for path, desc in items:
    p = PROJECT_ROOT / path.rstrip("/")
    marker = "OK " if p.exists() else "-- "
    print(f"  {marker}{path:35s} {desc}")

## 5. Shipped agents

The agents you'll use across the other notebooks all live under `data/agents/`. Each carries a `config.json` (training hyperparameters), an `agent.pt` (inference state dict), a `checkpoint.pt` (resume state with optimiser), a `train.log` (per-step textual log), and an `eval_results.csv` (in-training eval). Quick listing:

In [ ]:
import json

agents_dir = PROJECT_ROOT / "data" / "agents"
for agent_dir in sorted(agents_dir.iterdir()):
    if not agent_dir.is_dir():
        continue
    config_path = agent_dir / "config.json"
    if not config_path.exists():
        continue
    cfg = json.loads(config_path.read_text())
    arch = cfg.get("architecture", "?")
    steps = cfg.get("total_timesteps", 0)
    has_agent = (agent_dir / "agent.pt").exists()
    print(
        f"  {agent_dir.name:30s} arch={arch:25s} budget={steps / 1e6:>5.0f}M  agent.pt={has_agent}"
    )

## Next steps

Now that the environment is verified and you know where everything lives:

1. Open [`01_evaluate.ipynb`](01_evaluate.ipynb) to load `UECD-SingleMap-Best` and play it against a couple of opponents.
2. Then `02_train.ipynb` to train a small agent from scratch.
3. Then the rest, in any order: `03_tournament.ipynb`, `04_bc.ipynb`, `05_bench.ipynb`, `06_analysis.ipynb`.

Every notebook is self-contained from this point: they import what they need, but they all assume the install is good (this notebook's section 2 passes).